In [19]:
import torch
import numpy as np
from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor
import av
import matplotlib.pyplot as plt

In [18]:
!pip install av

   ---------------------------------------- 0.0/31.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.8 MB ? eta -:--:--
    --------------------------------------- 0.5/31.8 MB 3.4 MB/s eta 0:00:10
   - -------------------------------------- 1.3/31.8 MB 3.7 MB/s eta 0:00:09
   -- ------------------------------------- 1.8/31.8 MB 3.7 MB/s eta 0:00:09
   -- ------------------------------------- 2.1/31.8 MB 3.6 MB/s eta 0:00:09
   --- ------------------------------------ 3.1/31.8 MB 3.2 MB/s eta 0:00:09
   ---- ----------------------------------- 3.9/31.8 MB 3.4 MB/s eta 0:00:09
   ----- ---------------------------------- 4.5/31.8 MB 3.2 MB/s eta 0:00:09
   ------ --------------------------------- 5.2/31.8 MB 3.4 MB/s eta 0:00:08
   ------- -------------------------------- 6.0/31.8 MB 3.4 MB/s eta 0:00:08
   -------- ------------------------------- 6.8/31.8 MB 3.5 MB/s eta 0:00:08
   --------- ------------------------------ 7.9/31.8 MB 3.5 MB/s eta 0:00:07
   ----------

In [2]:
class_mapping = {
    "Abuse": 0, "Arrest": 1, "Arson": 2, "Assault": 3, "Burglary": 4,
    "Explosion": 5, "Fighting": 6, "Normal Videos": 7, "Road Accidents": 8,
    "Robbery": 9, "Shooting": 10, "Shoplifting": 11, "Stealing": 12, "Vandalism": 13
}
reverse_mapping = {v: k for k, v in class_mapping.items()}

In [15]:
model_name = "MCG-NJU/videomae-v2-large"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [21]:
import tensorflow as tf

# 1. Load your saved model
model = tf.keras.models.load_model('trained_model/crime_detection_model.h5')

# 2. Print the architecture to see its layers and parameters
print("--- Model Architecture ---")
model.summary()





--- Model Architecture ---
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed (TimeDist  (None, 30, 64, 64, 32)    896       
 ributed)                                                        
                                                                 
 time_distributed_1 (TimeDi  (None, 30, 32, 32, 32)    0         
 stributed)                                                      
                                                                 
 time_distributed_2 (TimeDi  (None, 30, 32, 32, 64)    18496     
 stributed)                                                      
                                                                 
 time_distributed_3 (TimeDi  (None, 30, 16, 16, 64)    0         
 stributed)                                                      
                                                                 
 time_distributed_4 (TimeDi

In [ ]:
def load_video_frames(video_path, num_frames=16, size=(224, 224)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        if i in frame_indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, size)
            frames.append(frame)

    cap.release()

    if len(frames) < num_frames:
        frames.extend([frames[-1]] * (num_frames - len(frames)))

    frames = np.stack(frames, axis=0)
    frames = torch.tensor(frames, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    return frames

In [13]:
video_path = "C:/Users/pouss/Documents/CSAI/Rawi-Vision/Face Recognition/videos/fight.mp4"


In [14]:
# Load and preprocess
video_tensor = load_video_frames(video_path)
video_tensor = video_tensor.unsqueeze(0).to(device)

# Inference
with torch.no_grad():
    outputs = model(video_tensor)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_label = torch.argmax(probs, dim=-1).item()

print(f"Predicted label: {reverse_mapping[predicted_label]}")
     

Predicted label: Abuse
